# Tangram: Spatial Gene Imputation (Pseudo-6K Benchmark)

Maps scRNA-seq cells onto CosMx spatial spots using Tangram, then projects
scRNA-seq gene expression onto the spatial data to impute the missing ~12K genes.

**Input**: WTX CosMx h5ad; 6K gene list; scRNA-seq reference  
**Output**: `half_ori_half_tangram_imp_log1p.csv` — combined 6K observed + imputed 12K genes


In [ ]:
import tangram as tg
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = "/path/to/cosmx/data"
SCRNA_DIR = "/path/to/scrna/data"
OUT_DIR   = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX CosMx data (log1p normalized)
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad")
adata.obs['fov'] = adata.obs['fov'].astype(int)
adata.obs_names  = adata.obs['fov_cell_id'].tolist()

# Train/test split by FOV
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

test_adata  = adata[test_cells, :].copy()
train_adata = adata[train_cells, :].copy()  # used as spatial reference

In [ ]:
# scRNA-seq reference (full transcriptome, used as atlas)
scrna = sc.read_h5ad(f"{SCRNA_DIR}/uc_atlas_harmony_integrated.h5ad")

In [ ]:
# 6K gene panel (observed genes; Tangram maps from the full transcriptome)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()

# Restrict spatial test data to 6K genes (pseudo-6K experiment)
test_adata_sub = test_adata[:, test_adata.var.index.isin(genes_6k)].copy()
print(f"Spatial genes: {test_adata_sub.n_vars:,}  |  scRNA genes: {scrna.n_vars:,}")

# Tangram preprocessing: identify overlap genes between spatial and scRNA
tg.pp_adatas(scrna, test_adata_sub, genes_6k)
print(f"Training genes: {len(scrna.uns['training_genes'])}  |  Overlap: {len(scrna.uns['overlap_genes'])}")

In [ ]:
# Map scRNA cells onto spatial spots (uniform density prior)
ad_map = tg.map_cells_to_space(
    scrna,
    test_adata_sub,
    mode="cells",
    density_prior="uniform",
    num_epochs=500,
    device="cuda",
)

In [ ]:
# Project full scRNA gene expression onto spatial cells
ad_ge = tg.project_genes(adata_map=ad_map, adata_sc=scrna)
ad_ge.var_names = scrna.var_names
ad_ge.obsm = test_adata_sub.obsm

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────

# Full imputed expression (all scRNA genes projected to spatial)
tan_pred = pd.DataFrame(ad_ge.X)
tan_pred.index   = ad_ge.obs['fov_cell_id'].tolist()
tan_pred.columns = [g.upper() for g in ad_ge.var.index.tolist()]

# Combine: keep 6K observed counts; impute remaining 12K
raw_6k = pd.DataFrame(test_adata_sub.X)
raw_6k.columns = [g.upper() for g in test_adata_sub.var_names.tolist()]
raw_6k.index   = test_adata_sub.obs['fov_cell_id'].tolist()

imputed_genes = [g for g in tan_pred.columns if g not in genes_6k]
combined = pd.concat([raw_6k, tan_pred[imputed_genes]], axis=1)
combined  = combined[tan_pred.columns]  # restore original gene order
combined.to_csv(f"{OUT_DIR}/half_ori_half_tangram_imp_log1p.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")

---
## Pseudo-1K Section

Same Tangram workflow applied to the pseudo-1K panel (only 1K genes observed; ~17K imputed).
Output used for the per-gene PCC/SCC heatmap in `01_imputation_comparison.ipynb`.


In [ ]:
# 1K gene panel
genes_1k = pd.read_csv(f"{DATA_DIR}/1k_actual_gene_names.csv")['0'].tolist()

# Restrict test data to 1K genes
test_adata_sub_1k = test_adata[:, test_adata.var.index.isin(genes_1k)].copy()
print(f"1K observed genes: {test_adata_sub_1k.n_vars}  |  Imputation targets: {test_adata.n_vars - test_adata_sub_1k.n_vars:,}")

tg.pp_adatas(train_adata, test_adata_sub_1k, genes_1k)

In [ ]:
# Map and project (pseudo-1K)
ad_map_1k = tg.map_cells_to_space(
    train_adata,
    test_adata_sub_1k,
    mode="cells",
    density_prior="uniform",
    num_epochs=500,
    device="cuda",
)
ad_ge_1k = tg.project_genes(adata_map=ad_map_1k, adata_sc=train_adata)
ad_ge_1k.var_names = train_adata.var_names
ad_ge_1k.obsm = test_adata_sub_1k.obsm

In [ ]:
# Export pseudo-1K results
tan_pred_1k = pd.DataFrame(ad_ge_1k.X)
tan_pred_1k.index   = ad_ge_1k.obs['fov_cell_id'].tolist()
tan_pred_1k.columns = [g.upper() for g in ad_ge_1k.var.index.tolist()]

raw_1k = pd.DataFrame(test_adata_sub_1k.X)
raw_1k.columns = [g.upper() for g in test_adata_sub_1k.var_names]
raw_1k.index   = test_adata_sub_1k.obs['fov_cell_id'].tolist()

imputed_1k = [g for g in tan_pred_1k.columns if g not in genes_1k]
combined_1k = pd.concat([raw_1k, tan_pred_1k[imputed_1k]], axis=1)
combined_1k = combined_1k[tan_pred_1k.columns]
combined_1k.to_csv(f"{OUT_DIR}/1k_ori_17k_tangram_imp_log1p.csv")
print(f"Exported pseudo-1K: {combined_1k.shape[0]:,} cells x {combined_1k.shape[1]:,} genes")